**Starting Local Redis**

`brew services start redis`

`brew services stop redis`

`brew services restart redis`

THEN

`redis-cli`

`FLUSHDB`

In [2]:
import pandas as pd
import redis

In [3]:
# Connect to Redis
r = redis.Redis(
    host = "localhost",
    port = 6379,
    decode_responses = True
)

In [4]:
r.ping()

True

In [5]:
DATA = "202609-cbs-tripdata-preprocessed.csv"

In [6]:
print("Existing rides:", len(r.keys("ride:*")))
print("Existing departures:", r.zcard("stations:departures"))
print("Existing routes:", r.zcard("routes:popular"))

Existing rides: 0
Existing departures: 0
Existing routes: 0


In [7]:
test_df = pd.read_csv(
    DATA,
    nrows=5,
    dtype={"start_station_id": "string", "end_station_id": "string"}
)

print(test_df[["ride_id", "start_station_id", "end_station_id"]])

            ride_id start_station_id end_station_id
0  3FA776BA68AAC5F2            31727          31727
1  0DDE689949EEE2AA            31727          31727
2  4B620586F5D84EC8            32001          32001
3  CB1C2CADBB6C64AD            31320          31320
4  E3F0AA83CFB29B45            31802          31829


In [8]:
CHUNK_SIZE = 5000

for chunk in pd.read_csv(
    DATA,
    chunksize=5000,
    dtype={"start_station_id": "string", "end_station_id": "string"}
):
    # Pipeline setup
    chunk = chunk.fillna("")
    pipeline = r.pipeline(transaction=False)

    # Chunk processing
    for row in chunk.to_dict("records"):
        ride_id = row["ride_id"]
        start_id = str(row["start_station_id"])
        end_id = str(row["end_station_id"])

        # Individual ride hashes
        pipeline.hset(
            f"ride:{ride_id}",
            mapping=row
        )

        # Skip station operations if either station is missing
        if not start_id or not end_id:
            continue

        # Individual station hashes
        pipeline.hset(
            f"station:{start_id}", # Starting station
            mapping={
                "name": row["start_station_name"],
                "latitude": row["start_lat"],
                "longitude": row["start_lng"]
            }
        )
        
        pipeline.hset(
            f"station:{end_id}", # Ending station
            mapping={
                "name": row["end_station_name"],
                "latitude": row["end_lat"],
                "longitude": row["end_lng"]
            }
        )

        # Set: Unique destinations from one origin
        pipeline.sadd(
            f"station:{start_id}:destinations",
            end_id
        )

        # Sorted set: Departure ranking
        pipeline.zincrby(
            "stations:departures",
            1,
            start_id
        )

        # Sorted set: Arrival ranking
        pipeline.zincrby(
            "stations:arrivals",
            1,
            end_id
        )

        # Sorted set: Route rankings
        route = f"{start_id}->{end_id}"

        pipeline.zincrby(
            "routes:popular",
            1,
            route
        )

    pipeline.execute()

    print(f"Loaded {len(chunk):,} rides")

print("Finished!")

C:\Users\Arsha\AppData\Local\Temp\ipykernel_3292\1863950962.py:9: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  chunk = chunk.fillna("")


Loaded 5,000 rides
Loaded 5,000 rides


C:\Users\Arsha\AppData\Local\Temp\ipykernel_3292\1863950962.py:9: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  chunk = chunk.fillna("")


Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides


C:\Users\Arsha\AppData\Local\Temp\ipykernel_3292\1863950962.py:9: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  chunk = chunk.fillna("")


Loaded 5,000 rides


... (40 outputs left out)

C:\Users\Arsha\AppData\Local\Temp\ipykernel_3292\1863950962.py:9: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  chunk = chunk.fillna("")


Loaded 5,000 rides


C:\Users\Arsha\AppData\Local\Temp\ipykernel_3292\1863950962.py:9: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  chunk = chunk.fillna("")


Loaded 1,808 rides
Finished!


In [9]:
ride_count = sum(1 for _ in r.scan_iter(match="ride:*"))
print("Total rides in Redis:", ride_count)

Total rides in Redis: 676808


In [10]:
r.bgsave()

True

In [11]:
print("Saving in progress:", r.info("persistence")["rdb_bgsave_in_progress"])
print("Last save status:", r.info("persistence")["rdb_last_bgsave_status"])

Saving in progress: 0
Last save status: ok


In [12]:
print("Redis connected:", r.ping())

ride_count = sum(1 for _ in r.scan_iter(match="ride:*"))
print("Total rides after restart:", ride_count)

Redis connected: True
Total rides after restart: 676808


In [13]:
print("Departure stations:", r.zcard("stations:departures"))
print("Arrival stations:", r.zcard("stations:arrivals"))
print("Popular routes:", r.zcard("routes:popular"))

print("Total departures:", sum(r.zrange("stations:departures", 0, -1, withscores=True)[i][1] for i in range(r.zcard("stations:departures"))))

Departure stations: 865
Arrival stations: 866
Popular routes: 85167
Total departures: 509133.0


In [14]:
df_check = pd.read_csv(DATA)

valid_rides = (
    df_check["start_station_id"].notna()
    & df_check["end_station_id"].notna()
).sum()

print("Rides with both station IDs:", valid_rides)

Rides with both station IDs: 509133


# Redis Queries

**Hash: Ride information**
- Get all information about one ride: `HGETALL ride:9FF60D31E16B6637`
- From that one ride, get the starting station name: `HGET ride:9FF60D31E16B6637 start_station_name`

**Hash: Station information**
- Get all information about one station: `HGETALL station:31623`
- Get the name from that one station: `HGET station:31623 name`

**Set: Get all destinations from one origin station**
- List all stations: `SMEMBERS station:31623:destinations`
- Count all statons: `SCARD station:31623:destinations`

**Sorted Set: Top 10 stations that riders arrive at**

**Sorted Set: Top 10 stations that riders depart from**

**Sorted Set: Top 10 routes taken by riders**